In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    
from config import *
print(f"Environment linked. Data dir: {DATA_DIR}")

In [ ]:
from scipy.stats import kurtosis
import h5py
import numpy as np

In [ ]:
class MachineA_Preprocessor:
    def __init__(self):
        self.f = h5py.File(RAW_DATA_FILE, 'r')
        self.splits = np.load(SPLIT_MAP_FILE)

    def process_and_save(self):
        # We only prepare train and validation sets now, test is saved for later
        self._process_subset('train', MACHINE_A_TRAIN_X, MACHINE_A_TRAIN_Y)
        self._process_subset('val',   MACHINE_A_VAL_X,   MACHINE_A_VAL_Y)

    def _process_subset(self, split_name, path_x, path_y):
        print(f"\n--- Processing {split_name.upper()} Set ---")
        indices = self.splits[split_name]
        
        # for now we only use a subset of data for speed
        limit = 10000
        if len(indices) > limit:
            print(f"sampling {limit} from {len(indices)} available samples.")
            # Sort first (for HDF5 speed), then take the first N
            indices = np.sort(indices)[:limit]
        else:
            indices = np.sort(indices)
        
        # Load Clean Data
        print("Loading Clean I/Q Data...")
        clean_iq = self.f['X'][indices]
        
        # Extract Features -> Label 0 (Clean)
        print("Extracting Features for CLEAN signals...")
        feats_clean = self._extract_features(clean_iq)
        labels_clean = np.zeros(len(feats_clean))
        
        # Generate Noise -> Label 1 (Noisy)
        print("Generating Barrage Noise & Extracting Features...")
        noisy_iq = self._add_barrage_noise(clean_iq)
        feats_noisy = self._extract_features(noisy_iq)
        labels_noisy = np.ones(len(feats_noisy))
        
        # Combine
        X = np.vstack([feats_clean, feats_noisy])
        Y = np.concatenate([labels_clean, labels_noisy])
        
        np.save(path_x, X)
        np.save(path_y, Y)
        print(f"Saved: {path_x.name} (Shape: {X.shape})")

    def _add_barrage_noise(self, iq_samples):
        # Barrage Interference (Broadband Gaussian Noise)
        n, l, _ = iq_samples.shape
        power = 10 ** (NOISE_POWER_DB / 10.0)
        noise = (np.random.normal(0, np.sqrt(power/2), (n, l)) + 
                 1j * np.random.normal(0, np.sqrt(power/2), (n, l)))
        return iq_samples + np.stack([noise.real, noise.imag], axis=2)

    def _extract_features(self, iq_batch):
        # Calculates: Mean Power, Variance, PAPR, Kurtosis
        # Convert to complex numbers for easier math
        complex_data = iq_batch[:, :, 0] + 1j * iq_batch[:, :, 1]
        
        # Mean Power
        power = np.abs(complex_data)**2
        mean_power = np.mean(power, axis=1)
                
        # PAPR (Peak-to-Average Power Ratio)
        peak = np.max(power, axis=1)
        papr = 10 * np.log10(peak / (mean_power + 1e-9))
        
        # Kurtosis
        kurt = kurtosis(np.abs(complex_data), axis=1)
        
        return np.column_stack((mean_power, papr, kurt))

    def close(self):
        self.f.close()

In [ ]:
processor = MachineA_Preprocessor()
processor.process_and_save()
processor.close()
print("\nDone!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Verifying Data Quality...")

X_train = np.load(MACHINE_A_TRAIN_X)
Y_train = np.load(MACHINE_A_TRAIN_Y)

df = pd.DataFrame(X_train, columns=MACHINE_A_FEATURES)
df['Label'] = Y_train 


fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Feature Distributions: Clean vs Noisy', fontsize=16)

axes = axes.flatten()

for i, feature in enumerate(MACHINE_A_FEATURES):
    # Plot Clean (Blue) and Noisy (Red)
    sns.histplot(data=df, x=feature, hue='Label', bins=50, ax=axes[i], 
                 kde=True, palette={0: 'blue', 1: 'red'}, alpha=0.5)
    
    axes[i].set_title(f"Distribution of {feature}")
    axes[i].set_xlabel(feature)
    axes[i].legend(title='Type', labels=['Noisy (1)', 'Clean (0)'])

plt.tight_layout()
plt.show()

# Correlation Matrix
# This shows if features are redundant (e.g., if Power and Variance are perfectly correlated)
plt.figure(figsize=(8, 6))
sns.heatmap(df.drop('Label', axis=1).corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()